# Step 4. 산불발생 공간·지형·접근성 최종 EDA

이 노트북은 `jsw/강원_재_EDA` 폴더의 마지막 EDA 단계다.

Step 1~3에서 기상, 캐나다 산불지수, 매칭 대조군, 선행 기상, 국지 임계치 후보를 이미 정리했으므로 Step 4에서는 공간·지형·토지피복·접근성과 공간 대조군만 확인한다.

기존 Step 5와 Step 6은 이 EDA 흐름에서 제거한다. 도로·임도·등산로·생활권·산림 내부 같은 인간활동 프록시 요소는 별도 원인 분석이 아니라 Step 4의 접근성·공간층 변수로 흡수한다.

결과 해석은 이 노트북에 작성하지 않는다. 표와 플롯을 함께 검토한 해석, 한계, 다음 반영사항은 `Step4_산불발생_공간지형및대조군_분석_진행예정로그.md`에만 기록한다.

In [ ]:
from pathlib import Path
import sys

NOTEBOOK_DIR = Path.cwd().resolve()
candidates = [NOTEBOOK_DIR, *NOTEBOOK_DIR.parents]
REPO_ROOT = next(
    (path for path in candidates if (path / "jsw/강원_재_EDA/re_eda_common.py").exists()),
    Path(r"D:/farm-system-public-02"),
)
MODULE_DIR = REPO_ROOT / "jsw/강원_재_EDA"
if str(MODULE_DIR) not in sys.path:
    sys.path.insert(0, str(MODULE_DIR))

from re_eda_common import (
    check_sources,
    configure_notebook,
    frame_inventory,
    load_access_lines,
    load_dem_metadata,
    load_fire,
    load_grid_bundle,
    load_infrastructure,
    load_landcover,
    load_roads,
    load_terrain,
)

configure_notebook()

## 1. 원천 파일 존재 여부

공간 EDA에 필요한 입력 파일만 확인한다. 캐나다 지수와 시간기상은 Step 1~3에서 이미 다뤘으므로 여기서는 다시 로딩하지 않는다.

In [ ]:
SOURCE_KEYS = [
    "fire",
    "weather_cells",
    "weather_grid",
    "climate_type",
    "terrain",
    "dem",
    "landcover",
    "roads",
    "trails",
    "forest_roads",
    "fire_stations",
    "fire_water",
]
source_audit = check_sources(SOURCE_KEYS)
display(source_audit)

## 2. 산불, 격자, 지형 자료 로딩

산불 발생지, 기상셀 격자, 기후지형유형, 기존 지형특성 계산값을 불러온다. DEM은 재추출보다 기존 계산값의 품질 감사에 우선 사용한다.

In [ ]:
fire, fire_points = load_fire()
grid_bundle = load_grid_bundle()
weather_cells = grid_bundle["cells"]
climate_type = grid_bundle["climate"]
weather_grid = grid_bundle["grid"]
terrain = load_terrain()
dem_metadata = load_dem_metadata()

print("산불 원자료 행 수:", len(fire))
print("산불 포인트 CRS:", fire_points.crs)
print("기상 격자 CRS:", weather_grid.crs)
print("기상셀 수:", len(weather_cells))
display(terrain.head())
display(dem_metadata)

## 3. 토지피복·도로 로딩

토지피복과 도로는 대용량 파일이므로 이후 실제 분석 셀에서는 필요한 범위 클리핑과 캐싱을 먼저 수행한다.

In [ ]:
landcover = load_landcover()
roads = load_roads()

print("토지피복:", len(landcover), "건 / CRS:", landcover.crs)
print("도로:", len(roads), "건 / CRS:", roads.crs)

## 4. 접근성 레이어 로딩

기존 Step 6에서 별도 인간활동 프록시로 다루려던 등산로·임도·생활권 요소는 이 단계에서 접근성 및 공간층 변수로만 사용한다.

In [ ]:
access_lines = load_access_lines()
trails = access_lines["trails"]
forest_roads = access_lines["forest_roads"]

infrastructure = load_infrastructure()
fire_stations = infrastructure["fire_stations"]
fire_water = infrastructure["fire_water"]

print("등산로:", len(trails), "건 / CRS:", trails.crs)
print("임도:", len(forest_roads), "건 / CRS:", forest_roads.crs)
print("소방서:", len(fire_stations), "건 / CRS:", fire_stations.crs)
print("소방용수:", len(fire_water), "건 / CRS:", fire_water.crs)

## 5. 로딩 결과 요약

In [ ]:
loaded_frames = {
    "fire": fire,
    "fire_points": fire_points,
    "weather_cells": weather_cells,
    "climate_type": climate_type,
    "weather_grid": weather_grid,
    "terrain": terrain,
    "landcover": landcover,
    "roads": roads,
    "trails": trails,
    "forest_roads": forest_roads,
    "fire_stations": fire_stations,
    "fire_water": fire_water,
}
display(frame_inventory(loaded_frames))

## 다음 구현 범위

1. 모든 공간자료의 CRS, bounds, invalid/empty geometry를 감사한다.
2. 산불 발생지에 토지피복·지형·접근성 변수를 결합한다.
3. `생활권-WUI`, `산림 접근권`, `산림 내부` 공간층을 만든다.
4. 동일 기상셀 또는 층화 공간 대조군 후보풀을 생성하고 편향 여부를 확인한다.
5. Step 4 결과를 최종 EDA 요약으로 정리한다.

Step 5와 Step 6은 이 EDA 폴더에서 진행하지 않는다. 후속 모델링은 별도 폴더에서 새 계획으로 시작한다.